# SpO2 / HB diagnostics

Interactive view of psgscoring's HB pipeline on the 6 synthetic
golden cases. **Case picker** is at the top; the **event prev / next
controls** live inside section 2 (since only section 2 depends on
which event is selected).

**Sections**

1. **Signal + event overlay** — flow, effort, SpO2 with synthetic
   ground-truth events (filled) and pipeline-detected events
   (outlined). Colour = event kind; legend shows kinds present in
   the current case.
2. **HB by method (per-event focus)** — one subplot per registered
   method, all zoomed to the currently-selected event (2× event
   duration window). Each subplot shows the integration window,
   the per-event baseline, and the deficit area integrated to
   compute hypoxic burden. The event prev / next buttons and the
   Event dropdown step through detected events; the selection is
   shared across methods.
3. **Ensemble search window** — the ensemble-averaged SpO2 curve
   used by `baseline_method='ensemble'`, with the resolved window
   shaded. Visualises psgscoring issue #4 (window collapse on
   clustered events).
4. **Per-event baselines** — distribution of per-event baseline
   values for each method.

Run all cells. The case picker, header summary, and all four
sections appear below the last cell.

In [ ]:
%matplotlib inline
import sys, warnings
from pathlib import Path

# Find the repo root by walking up from the notebook's cwd looking
# for the psgscoring package. Works whether the notebook is opened
# from `binder/`, `notebooks/`, or the repo root itself.
def _find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'psgscoring' / '__init__.py').exists():
            return candidate
    raise RuntimeError(f'psgscoring package not found above {Path.cwd()}')

_repo = _find_repo_root()
sys.path.insert(0, str(_repo))
sys.path.insert(0, str(_repo / 'tests'))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

import psgscoring
from psgscoring.spo2 import (
    HB_BASELINE_METHODS, compute_hypoxic_burden, _ensemble_search_window,
)
from test_golden_output import CASES, make_raw, _arousals_for, hb_by_method

import logging
logging.getLogger('psgscoring').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

# Colour palette for event types. Synth and detected share colours
# (filled vs outlined distinguishes the two channels).
COLOR_BY_KIND = {
    'obstructive':     '#d62728',
    'central':         '#1f77b4',
    'hypopnea':        '#2ca02c',
    'hypopnea_mixed':  '#bcbd22',
    'mixed':           '#ff7f0e',
}
def _color(kind):
    return COLOR_BY_KIND.get(kind, '#7f7f7f')

In [ ]:
def load_case(name):
    """Build the synthetic case and run the full pipeline.
    Returns (cfg, raw, hypno, cmap, pipeline_output).
    """
    cfg = dict(CASES[name])
    profile = cfg.pop('profile')
    use_arousals = cfg.pop('arousals', False)
    raw, hypno, cmap = make_raw(**cfg)
    kwargs = dict(channel_map=cmap, scoring_profile=profile)
    if use_arousals:
        kwargs['arousal_events'] = _arousals_for(cfg['events'])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        out = psgscoring.run_pneumo_analysis(raw, hypno, **kwargs)
    return cfg, raw, hypno, cmap, out

In [ ]:
def plot_signals(cfg, raw, out):
    """Section 1: signals with synth (filled) + detected (outlined) events."""
    sf = raw.info['sfreq']
    n = raw.n_times
    t = np.arange(n) / sf
    channels = ['FLOW', 'THORAX', 'ABDOMEN', 'SPO2']
    synth = cfg.get('events', [])
    det = out.get('respiratory', {}).get('events', []) or []

    fig, axes = plt.subplots(4, 1, sharex=True, figsize=(13, 7.5))
    for ax, ch in zip(axes, channels):
        try:
            data = raw.get_data(picks=ch)[0]
        except Exception:
            ax.text(0.5, 0.5, f'{ch}: unavailable', ha='center',
                    va='center', transform=ax.transAxes)
            continue
        ax.plot(t, data, lw=0.4, color='black')
        ax.set_ylabel(ch)
        for t0, t1, kind in synth:
            ax.axvspan(t0, t1, alpha=0.18, color=_color(kind), lw=0)
        for d in det:
            d_t0 = float(d.get('onset_s', 0))
            d_t1 = d_t0 + float(d.get('duration_s', 0))
            ax.axvspan(d_t0, d_t1, facecolor='none',
                       edgecolor=_color(d.get('type')), lw=1.2)

    axes[-1].set_xlabel('time (s)')

    # Build the legend in two halves:
    #  - kinds present in this case (coloured swatches)
    #  - synth vs detected channel distinction (filled vs outlined)
    kinds_present = []
    for _, _, k in synth:
        if k not in kinds_present: kinds_present.append(k)
    for d in det:
        k = d.get('type')
        if k and k not in kinds_present: kinds_present.append(k)

    handles = [plt.Rectangle((0, 0), 1, 1, alpha=0.5, color=_color(k), label=k)
               for k in kinds_present]
    handles += [
        plt.Rectangle((0, 0), 1, 1, alpha=0.18, color='#666', label='synth (input)'),
        plt.Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='#666', lw=1.2,
                      label='detected'),
    ]
    axes[0].legend(handles=handles, loc='upper right', framealpha=0.9,
                   fontsize=8, ncol=2)
    fig.suptitle('Synthetic events (filled) vs detected events (outlined). '
                 'Colour = event kind.')
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_hb_per_method(raw, hypno, out, cmap, event_idx):
    """Section 2: one subplot per HB baseline method, all focused on the
    same detected event (selected by event_idx). Each subplot shows:

    - the SpO2 trace zoomed to the union of (2× the event duration
      around the event) and (every method's integration window with a
      small padding) — so when the ensemble method's degenerate window
      lands far from the event (issue #4), it's still visible;
    - the event itself shaded by kind colour;
    - the method's integration window (blue outline);
    - the method's per-event baseline (red dashed line);
    - the deficit area (red shading between baseline and SpO2 within
      the integration window) — this is what gets integrated to
      compute area.

    All methods share the same x-axis for easy comparison.
    """
    events = out.get('respiratory', {}).get('events', []) or []
    if not events:
        display(HTML('<p><i>No detected events — nothing to plot.</i></p>'))
        return
    if event_idx is None or event_idx < 0 or event_idx >= len(events):
        event_idx = 0
    ev = events[event_idx]
    ev_t0 = float(ev['onset_s'])
    ev_t1 = ev_t0 + float(ev['duration_s'])
    ev_dur = float(ev['duration_s'])
    # Default zoom: 2× event duration, centred on event.
    base_vis = (ev_t0 - ev_dur / 2, ev_t1 + ev_dur / 2)

    spo2 = raw.get_data(picks=cmap['spo2'])[0]
    sf = raw.info['sfreq']
    t = np.arange(len(spo2)) / sf

    methods = list(HB_BASELINE_METHODS)

    # First pass: run every method and find its per-event record.
    results = {}
    for method in methods:
        r = compute_hypoxic_burden(
            spo2, sf, events, hypno,
            baseline_method=method, return_diagnostics=True,
        ) or {}
        per_event = r.get('per_event') or []
        rec = None
        if per_event:
            rec = min(per_event,
                      key=lambda p: abs(p.get('onset_s', 0) - ev_t0))
            if abs(rec.get('onset_s', 0) - ev_t0) > 0.5:
                rec = None
        results[method] = (r, rec)

    # Shared visible range: union of base 2× window AND every
    # method's integration window with a small padding.
    vis_t0, vis_t1 = base_vis
    for _, rec in results.values():
        if rec is None:
            continue
        ws = float(rec['win_start_s'])
        we = float(rec['win_end_s'])
        pad = max((we - ws) * 0.25, 2.0)
        vis_t0 = min(vis_t0, ws - pad)
        vis_t1 = max(vis_t1, we + pad)
    mask = (t >= vis_t0) & (t <= vis_t1)

    fig, axes = plt.subplots(len(methods), 1, sharex=True,
                             figsize=(12, 3.0 * len(methods)))
    if len(methods) == 1:
        axes = [axes]

    for ax, method in zip(axes, methods):
        r, rec = results[method]
        ax.plot(t[mask], spo2[mask], color='black', lw=0.8)
        ax.axvspan(ev_t0, ev_t1, alpha=0.15, color=_color(ev.get('type')),
                   label=f'event ({ev.get("type")})')

        title_bits = [f'{method}: hb={r.get("hypoxic_burden")}']
        if rec is None:
            title_bits.append('event skipped (no per_event record)')
        else:
            bl = float(rec['baseline'])
            ws = float(rec['win_start_s'])
            we = float(rec['win_end_s'])
            area = float(rec['area'])

            ax.axvspan(ws, we, facecolor='none', edgecolor='#1f77b4',
                       lw=1.5,
                       label=f'integration window [{ws:.1f}, {we:.1f}]')
            ax.axhline(bl, color='#d62728', lw=1.0, linestyle='--',
                       label=f'baseline {bl:.1f}%')
            seg_mask = mask & (t >= ws) & (t <= we)
            if seg_mask.any():
                seg_t, seg_v = t[seg_mask], spo2[seg_mask]
                ax.fill_between(seg_t, bl, seg_v,
                                where=(seg_v < bl) & (~np.isnan(seg_v)),
                                alpha=0.25, color='#d62728', linewidth=0,
                                label=f'deficit (area={area:.1f} %·s)')
            title_bits.append(f'event area={area:.1f} %·s')

        ax.set_ylabel('SpO2 (%)')
        ax.set_title(' — '.join(title_bits))
        ax.legend(fontsize=8, loc='lower right')
        ax.grid(alpha=0.2)

    axes[0].set_xlim(vis_t0, vis_t1)  # shared via sharex=True
    axes[-1].set_xlabel('time (s)')
    fig.suptitle(f'Event #{event_idx}/{len(events)-1}: {ev.get("type")} '
                 f'at t=[{ev_t0:.1f}, {ev_t1:.1f}] (dur={ev_dur:.1f}s)')
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_ensemble(raw, hypno, out, cmap):
    """Section 3: ensemble curve + resolved window (issue #4 visual)."""
    events = out.get('respiratory', {}).get('events', []) or []
    if len(events) < 3:
        print('< 3 detected events — ensemble fallback would fire; nothing to plot.')
        return
    spo2 = raw.get_data(picks=cmap['spo2'])[0]
    sf = raw.info['sfreq']
    left, right, curve, t_axis = _ensemble_search_window(
        spo2, sf, events, pre_s=60.0, post_s=60.0,
    )
    if left is None:
        print('_ensemble_search_window returned None (fewer than 3 usable segments).')
        return

    fig, ax = plt.subplots(figsize=(12, 4.0))
    ax.plot(t_axis, curve, color='black', lw=1.2,
            label='ensemble-averaged SpO2 (3-s MA)')
    ax.axvspan(left, right, alpha=0.18, color='#d62728',
               label=f'resolved window [{left:+.1f}, {right:+.1f}] s')
    ax.axvline(0, color='gray', lw=0.6, linestyle='--')
    nadir_idx = int(np.nanargmin(curve))
    ax.axvline(t_axis[nadir_idx], color='#1f77b4', lw=0.8, linestyle=':',
               label=f'nadir @ {t_axis[nadir_idx]:+.1f} s')
    ax.set_xlabel('seconds relative to event termination (t=0)')
    ax.set_ylabel('SpO2 (%)')
    ax.set_title('Ensemble search window — issue #4 fingerprint: '
                 'window pinned to the −60 s curve limit on clustered events.')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_baselines(raw, hypno, out, cmap):
    """Section 4: per-event baseline distribution, per method."""
    events = out.get('respiratory', {}).get('events', []) or []
    spo2 = raw.get_data(picks=cmap['spo2'])[0]
    sf = raw.info['sfreq']

    fig, ax = plt.subplots(figsize=(8, 3.5))
    for i, method in enumerate(HB_BASELINE_METHODS):
        r = compute_hypoxic_burden(
            spo2, sf, events, hypno,
            baseline_method=method, return_diagnostics=True,
        ) or {}
        bls = [p['baseline'] for p in (r.get('per_event') or [])
               if isinstance(p.get('baseline'), (int, float))]
        if not bls:
            continue
        # Strip plot: jitter on x = method index.
        x = np.full(len(bls), i) + np.random.default_rng(42).normal(0, 0.05, len(bls))
        ax.scatter(x, bls, alpha=0.75, edgecolor='black', s=60,
                   label=f'{method} (n={len(bls)}, median={np.median(bls):.1f})')
    sleep_spo2 = spo2[~np.isnan(spo2)]
    if len(sleep_spo2) > 10:
        ref = float(np.percentile(sleep_spo2, 95))
        ax.axhline(ref, color='gray', lw=0.8, linestyle='--',
                   label=f'global 95-pct reference ({ref:.1f}%)')
    ax.set_xticks(range(len(HB_BASELINE_METHODS)))
    ax.set_xticklabels(HB_BASELINE_METHODS)
    ax.set_ylabel('per-event baseline (%)')
    ax.set_title('Per-event baselines — issue #4: ensemble baseline often '
                 'lands below the recovered SpO2 line.')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Widgets + per-section Output containers ─────────────────────────────
case_picker = widgets.Dropdown(
    options=list(CASES), value='hypopnea_clean',
    description='Case:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='320px'),
)
event_picker = widgets.Dropdown(
    description='Event:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='420px'),
)
prev_btn = widgets.Button(description='◀ prev',
                          layout=widgets.Layout(width='80px'))
next_btn = widgets.Button(description='next ▶',
                          layout=widgets.Layout(width='80px'))

out_header = widgets.Output()
out_sec1 = widgets.Output()
out_sec2 = widgets.Output()
out_sec3 = widgets.Output()
out_sec4 = widgets.Output()

_case_cache = {}

def _load_cached(case):
    if case not in _case_cache:
        _case_cache[case] = load_case(case)
    return _case_cache[case]


def _redraw_section2(*_):
    """Cheap: only re-renders section 2 (fast event-stepping)."""
    case = case_picker.value
    cfg, raw, hypno, cmap, out = _load_cached(case)
    with out_sec2:
        out_sec2.clear_output(wait=True)
        plot_hb_per_method(raw, hypno, out, cmap, event_picker.value)


def _redraw_all(*_):
    """Heavy: re-renders every section. Triggered when case changes."""
    case = case_picker.value
    cfg, raw, hypno, cmap, out = _load_cached(case)
    resp = out.get('respiratory', {}) or {}
    rs = resp.get('summary', {}) or {}
    sp = (out.get('spo2', {}) or {}).get('summary', {}) or {}
    cq = out.get('channel_quality', {}) or {}
    events = resp.get('events', []) or []

    with out_header:
        out_header.clear_output(wait=True)
        print(f'=== {case} ===')
        print(f'  resp: ahi={rs.get("ahi_total")} n={len(events)}'
              f' grade={cq.get("overall_grade")}')
        print(f'  spo2: odi_3pct={sp.get("odi_3pct")}'
              f' odi_4pct={sp.get("odi_4pct")}'
              f' min={sp.get("min_spo2")} n_desat={sp.get("n_desaturations")}')

    # Rebuild event_picker options without triggering its observer
    # (we want a single section-2 redraw at the end of this function).
    event_picker.unobserve(_redraw_section2, names='value')
    opts = [(f'#{i}: t=[{ev["onset_s"]:.1f}, '
             f'{ev["onset_s"]+ev["duration_s"]:.1f}] '
             f'({ev.get("type", "?")})', i)
            for i, ev in enumerate(events)]
    event_picker.options = opts
    if opts:
        event_picker.value = 0
    event_picker.observe(_redraw_section2, names='value')

    with out_sec1:
        out_sec1.clear_output(wait=True)
        plot_signals(cfg, raw, out)
    with out_sec3:
        out_sec3.clear_output(wait=True)
        plot_ensemble(raw, hypno, out, cmap)
    with out_sec4:
        out_sec4.clear_output(wait=True)
        plot_baselines(raw, hypno, out, cmap)

    # Section 2 last — uses event_picker.value which we just reset.
    _redraw_section2()


def _step(delta):
    opts = event_picker.options
    if not opts:
        return
    vals = [v for _, v in opts]
    cur = event_picker.value
    if cur not in vals:
        event_picker.value = vals[0]
        return
    i = vals.index(cur)
    j = max(0, min(len(vals) - 1, i + delta))
    if j != i:
        event_picker.value = vals[j]


case_picker.observe(_redraw_all, names='value')
event_picker.observe(_redraw_section2, names='value')
prev_btn.on_click(lambda _: _step(-1))
next_btn.on_click(lambda _: _step(+1))

# Initial render
_redraw_all()

# ── Layout: case picker at the top; event controls live inside section 2.
display(case_picker)
display(out_header)
display(HTML('<h3>Section 1 — signals + events</h3>'))
display(out_sec1)
display(HTML('<h3>Section 2 — HB by method (per-event focus)</h3>'))
display(widgets.HBox([prev_btn, event_picker, next_btn]))
display(out_sec2)
display(HTML('<h3>Section 3 — ensemble search window</h3>'))
display(out_sec3)
display(HTML('<h3>Section 4 — per-event baseline distribution</h3>'))
display(out_sec4)